In [0]:
# ============================================================
# 05_EDA_BUSINESS_INSIGHTS
# Gold → SQL Analytics → Key Business Findings
# ============================================================

GOLD_TABLE = "workspace.default.gold_loan_features"
PRED_TABLE = "workspace.default.gold_loan_predictions"

df = spark.table(GOLD_TABLE)
print("✅ Rows:", df.count())
print("✅ Columns:", len(df.columns))

✅ Rows: 307511
✅ Columns: 89


In [0]:
print("📊 1. Default Rate by Contract Type:")
spark.sql(f"""
    SELECT 
        NAME_CONTRACT_TYPE,
        COUNT(*) AS total_applicants,
        SUM(TARGET) AS total_defaults,
        ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) AS default_rate_pct
    FROM {GOLD_TABLE}
    GROUP BY NAME_CONTRACT_TYPE
    ORDER BY default_rate_pct DESC
""").show()

📊 1. Default Rate by Contract Type:
+------------------+----------------+--------------+----------------+
|NAME_CONTRACT_TYPE|total_applicants|total_defaults|default_rate_pct|
+------------------+----------------+--------------+----------------+
|        Cash loans|          278232|         23221|            8.35|
|   Revolving loans|           29279|          1604|            5.48|
+------------------+----------------+--------------+----------------+



In [0]:
print("📊 2. Default Rate by Education Level:")
spark.sql(f"""
    SELECT 
        NAME_EDUCATION_TYPE,
        COUNT(*) AS total_applicants,
        SUM(TARGET) AS total_defaults,
        ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) AS default_rate_pct
    FROM {GOLD_TABLE}
    GROUP BY NAME_EDUCATION_TYPE
    ORDER BY default_rate_pct DESC
""").show()

📊 2. Default Rate by Education Level:
+--------------------+----------------+--------------+----------------+
| NAME_EDUCATION_TYPE|total_applicants|total_defaults|default_rate_pct|
+--------------------+----------------+--------------+----------------+
|     Lower secondary|            3816|           417|           10.93|
|Secondary / secon...|          218391|         19524|            8.94|
|   Incomplete higher|           10277|           872|            8.48|
|    Higher education|           74863|          4009|            5.36|
|     Academic degree|             164|             3|            1.83|
+--------------------+----------------+--------------+----------------+



In [0]:
print("📊 3. Default Rate by Income Type:")
spark.sql(f"""
    SELECT 
        NAME_INCOME_TYPE,
        COUNT(*) AS total_applicants,
        SUM(TARGET) AS total_defaults,
        ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) AS default_rate_pct
    FROM {GOLD_TABLE}
    GROUP BY NAME_INCOME_TYPE
    ORDER BY default_rate_pct DESC
""").show()

📊 3. Default Rate by Income Type:
+--------------------+----------------+--------------+----------------+
|    NAME_INCOME_TYPE|total_applicants|total_defaults|default_rate_pct|
+--------------------+----------------+--------------+----------------+
|     Maternity leave|               5|             2|           40.00|
|          Unemployed|              22|             8|           36.36|
|             Working|          158774|         15224|            9.59|
|Commercial associate|           71617|          5360|            7.48|
|       State servant|           21703|          1249|            5.75|
|           Pensioner|           55362|          2982|            5.39|
|             Student|              18|             0|            0.00|
|         Businessman|              10|             0|            0.00|
+--------------------+----------------+--------------+----------------+



In [0]:
print("📊 4. Default Rate by Age Group:")
spark.sql(f"""
    SELECT 
        CASE 
            WHEN AGE_YEARS < 25 THEN 'Under 25'
            WHEN AGE_YEARS < 35 THEN '25-34'
            WHEN AGE_YEARS < 45 THEN '35-44'
            WHEN AGE_YEARS < 55 THEN '45-54'
            ELSE '55+'
        END AS age_group,
        COUNT(*) AS total_applicants,
        SUM(TARGET) AS total_defaults,
        ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) AS default_rate_pct
    FROM {GOLD_TABLE}
    GROUP BY age_group
    ORDER BY default_rate_pct DESC
""").show()

📊 4. Default Rate by Age Group:
+---------+----------------+--------------+----------------+
|age_group|total_applicants|total_defaults|default_rate_pct|
+---------+----------------+--------------+----------------+
| Under 25|           11923|          1474|           12.36|
|    25-34|           72152|          7706|           10.68|
|    35-44|           84231|          7085|            8.41|
|    45-54|           70042|          4951|            7.07|
|      55+|           69163|          3609|            5.22|
+---------+----------------+--------------+----------------+



In [0]:
print("📊 5. Default Rate by Credit Amount Bracket:")
spark.sql(f"""
    SELECT 
        CASE 
            WHEN AMT_CREDIT < 200000  THEN 'Low (<200K)'
            WHEN AMT_CREDIT < 500000  THEN 'Medium (200K-500K)'
            WHEN AMT_CREDIT < 1000000 THEN 'High (500K-1M)'
            ELSE 'Very High (>1M)'
        END AS credit_bracket,
        COUNT(*) AS total_applicants,
        SUM(TARGET) AS total_defaults,
        ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) AS default_rate_pct,
        ROUND(AVG(AMT_CREDIT), 0) AS avg_credit_amount
    FROM {GOLD_TABLE}
    GROUP BY credit_bracket
    ORDER BY default_rate_pct DESC
""").show()

📊 5. Default Rate by Credit Amount Bracket:
+------------------+----------------+--------------+----------------+-----------------+
|    credit_bracket|total_applicants|total_defaults|default_rate_pct|avg_credit_amount|
+------------------+----------------+--------------+----------------+-----------------+
|Medium (200K-500K)|          113189|         10115|            8.94|         331075.0|
|    High (500K-1M)|          108193|          9288|            8.58|         702287.0|
|       Low (<200K)|           36144|          2490|            6.89|         142386.0|
|   Very High (>1M)|           49985|          2932|            5.87|        1312477.0|
+------------------+----------------+--------------+----------------+-----------------+



In [0]:
print("📊 6. Full Risk Segment Summary:")
spark.sql(f"""
    SELECT 
        RISK_SEGMENT,
        COUNT(*) AS total_applicants,
        SUM(TARGET) AS total_defaults,
        ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) AS default_rate_pct,
        ROUND(AVG(AMT_CREDIT), 0) AS avg_credit,
        ROUND(AVG(AMT_INCOME_TOTAL), 0) AS avg_income,
        ROUND(AVG(EXT_SOURCE_MEAN), 4) AS avg_ext_score
    FROM {GOLD_TABLE}
    GROUP BY RISK_SEGMENT
    ORDER BY default_rate_pct DESC
""").show()

📊 6. Full Risk Segment Summary:
+--------------+----------------+--------------+----------------+----------+----------+-------------+
|  RISK_SEGMENT|total_applicants|total_defaults|default_rate_pct|avg_credit|avg_income|avg_ext_score|
+--------------+----------------+--------------+----------------+----------+----------+-------------+
|VERY_HIGH_RISK|           24249|          4828|           19.91|  496669.0|  147537.0|        0.123|
|     HIGH_RISK|           89053|         10149|           11.40|  566196.0|  163830.0|       0.3146|
|   MEDIUM_RISK|          116955|          7703|            6.59|  606420.0|  172136.0|       0.5059|
|      LOW_RISK|           77254|          2145|            2.78|  657805.0|  176145.0|       0.6695|
+--------------+----------------+--------------+----------------+----------+----------+-------------+



In [0]:
print("""
╔══════════════════════════════════════════════════════════╗
║         LOAN DEFAULT PREDICTION — KEY INSIGHTS           ║
╠══════════════════════════════════════════════════════════╣
║  Dataset   : 307,511 applicants | 122 raw features       ║
║  Defaults  : 24,825 (8.1%) — imbalanced dataset          ║
║  Pipeline  : Bronze → Silver → Gold (Medallion)          ║
║  Features  : 75 after cleaning + 8 engineered features   ║
╠══════════════════════════════════════════════════════════╣
║  MODEL PERFORMANCE                                       ║
║  Algorithm : Random Forest (class_weight={0:1, 1:10})    ║
║  ROC-AUC   : 0.7345                                      ║
║  Key Features: EXT_SOURCE_MEAN, AMT_CREDIT,              ║
║                CREDIT_INCOME_RATIO, AGE_YEARS            ║
╠══════════════════════════════════════════════════════════╣
║  BUSINESS FINDINGS                                       ║
║  • Very High Risk segment has highest default rate       ║
║  • Younger applicants default more than older ones       ║
║  • Lower education = higher default risk                 ║
║  • Cash loans default more than revolving loans          ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║         LOAN DEFAULT PREDICTION — KEY INSIGHTS           ║
╠══════════════════════════════════════════════════════════╣
║  Dataset   : 307,511 applicants | 122 raw features       ║
║  Defaults  : 24,825 (8.1%) — imbalanced dataset          ║
║  Pipeline  : Bronze → Silver → Gold (Medallion)          ║
║  Features  : 75 after cleaning + 8 engineered features   ║
╠══════════════════════════════════════════════════════════╣
║  MODEL PERFORMANCE                                       ║
║  Algorithm : Random Forest (class_weight={0:1, 1:10})    ║
║  ROC-AUC   : 0.7345                                      ║
║  Key Features: EXT_SOURCE_MEAN, AMT_CREDIT,              ║
║                CREDIT_INCOME_RATIO, AGE_YEARS            ║
╠══════════════════════════════════════════════════════════╣
║  BUSINESS FINDINGS                                       ║
║  • Very High Risk segment has highest default rate       ║
║  • Younger applicants

In [0]:
# Show all versions of Gold table — proves Delta Lake lineage
print("📊 Delta Lake Time Travel — Gold Table History:")
spark.sql(f"DESCRIBE HISTORY {GOLD_TABLE}").select(
    "version", "timestamp", "operation", "operationParameters"
).show(10, truncate=False)

📊 Delta Lake Time Travel — Gold Table History:
+-------+-------------------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation                        |operationParameters                                                                                                                                                                                                                                                                                                                                        |
+-------+-------------------+---------------------------------+------------------------------------------------------------------